
# ANSC 4040 — Optimized Hybrid HGB Pipeline

This notebook is the cleaned final version of the AnimalId recovery pipeline.

**Final retained model**
- Global structural candidate generation: **Top 320**
- Reranker candidate set: **Top 160**
- Candidate-level trajectory / prototype / distance / rank features: **52**
- Final reranker: **HistGradientBoostingClassifier (HGB)**
- Training data: **Fresh multi-snapshot + long-gap multi-snapshot**
- Outer validation reference: **training data only, cutoff 2020-12-13**

Exploratory branches that did not improve the final outer-validation result (XGBRanker, two-stage HGB, tolerance-band features, ReferenceAgeDays, and ensemble/oracle experiments) have been removed from the executable pipeline.

The notebook intentionally leaves the outer **test split untouched**.


In [2]:

# ============================================================
# 1. Imports and final configuration
# ============================================================

from pathlib import Path
import gc
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42

MODEL_DATA_PATH = Path(
    r"/Library/ansci-4040-fall-2026/jl4937_6040-project-1/dataset/Dataset_for_model_buliding_2.csv"
)

# Only columns used by the retained final pipeline.
MODEL_COLUMNS = [
    "AnimalId",
    "LactationNumber",
    "DaysInMilk",
    "ReproductionStatus",
    "EventDate",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
]

DIM_BIN_WIDTH = 7
REFERENCE_ROWS_PER_GROUP = 250

GLOBAL_MAX_CANDIDATES = 320
MODEL_TOP_K = 160
PROJECTED_RANK_PENALTY_DAYS = 7
DIM_TOLERANCE_DAYS = 30

# Legacy directory name retained so the optimized notebook can reuse
# candidate caches created during development.
CACHE_DIR = Path("xgboost_cache")
CACHE_DIR.mkdir(exist_ok=True)

FRESH_SNAPSHOT_DATES = [
    pd.Timestamp("2020-07-10"),
    pd.Timestamp("2020-07-20"),
    pd.Timestamp("2020-07-31"),
    pd.Timestamp("2020-08-10"),
    pd.Timestamp("2020-08-20"),
]

# Fixed during development to match the internal DEV last-seen distribution.
FRESH_TARGET_BIN_COUNTS = {
    "00_0_14": 280,
    "01_15_30": 156,
    "02_31_60": 756,
    "03_61_90": 500,
    "04_Over90": 8,
    "MissingSameLactation": 300,
}

LONGGAP_SNAPSHOT_DATES = [
    pd.Timestamp("2020-04-01"),
    pd.Timestamp("2020-05-01"),
    pd.Timestamp("2020-06-01"),
    pd.Timestamp("2020-07-01"),
]

# Fixed to reproduce the reference-age distribution observed in the
# 2,000-query outer-validation sample.
LONGGAP_TARGETS = {
    "1-30 days": 394,
    "31-60 days": 375,
    "61-90 days": 384,
    "91-120 days": 361,
    "121-160 days": 486,
}

LONGGAP_BOUNDS = {
    "1-30 days": (1, 30),
    "31-60 days": (31, 60),
    "61-90 days": (61, 90),
    "91-120 days": (91, 120),
    "121-160 days": (121, 160),
}

OUTER_VALIDATION_SAMPLE_SIZE = 2000

HGB_PARAMS = {
    "learning_rate": 0.05,
    "max_iter": 400,
    "max_leaf_nodes": 7,
    "min_samples_leaf": 10,
    "l2_regularization": 1.0,
    "early_stopping": True,
    "validation_fraction": 0.10,
    "n_iter_no_change": 20,
    "random_state": RANDOM_STATE,
}

print("Final candidate pipeline: Global Top-320 -> HGB Top-160")
print("Cache directory:", CACHE_DIR.resolve())


Final candidate pipeline: Global Top-320 -> HGB Top-160
Cache directory: /Library/ansci-4040-fall-2026/jl4937_6040-project-1/code/xgboost_cache


In [3]:

# ============================================================
# 2. Load data and create chronological outer/internal splits
# ============================================================

if not MODEL_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Model data file not found: {MODEL_DATA_PATH}\n"
        "Update MODEL_DATA_PATH in the configuration cell."
    )

data = pd.read_csv(MODEL_DATA_PATH, low_memory=False)
data.columns = data.columns.astype(str).str.strip()

missing_columns = [c for c in MODEL_COLUMNS if c not in data.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

data = data[MODEL_COLUMNS].copy()

data["AnimalId"] = data["AnimalId"].astype("string")
data["EventDate"] = pd.to_datetime(data["EventDate"], errors="coerce")

for col in ["LactationNumber", "DaysInMilk", "DurationSession_sec"]:
    data[col] = pd.to_numeric(data[col], errors="coerce", downcast="integer")

for col in [
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
]:
    data[col] = pd.to_numeric(data[col], errors="coerce", downcast="float")

data["ReproductionStatus"] = data["ReproductionStatus"].astype("category")

labelled_data = (
    data.loc[data["AnimalId"].notna() & data["EventDate"].notna()]
    .sort_values("EventDate")
    .reset_index(drop=True)
)

# ---------- Outer chronological 60/20/20 split ----------
n = len(labelled_data)
train_cut_date = labelled_data.iloc[int(n * 0.60)]["EventDate"]
validation_cut_date = labelled_data.iloc[int(n * 0.80)]["EventDate"]

data_train = labelled_data.loc[
    labelled_data["EventDate"] < train_cut_date
].copy()

data_validation = labelled_data.loc[
    (labelled_data["EventDate"] >= train_cut_date)
    & (labelled_data["EventDate"] < validation_cut_date)
].copy()

data_test = labelled_data.loc[
    labelled_data["EventDate"] >= validation_cut_date
].copy()

# ---------- Internal split inside OUTER TRAIN only ----------
reranker_base = data_train.sort_values("EventDate").reset_index(drop=True)
n_reranker = len(reranker_base)

history_cut_date = reranker_base.iloc[int(n_reranker * 0.70)]["EventDate"]
dev_cut_date = reranker_base.iloc[int(n_reranker * 0.85)]["EventDate"]

reranker_history = reranker_base.loc[
    reranker_base["EventDate"] < history_cut_date
].copy()

reranker_train_queries = reranker_base.loc[
    (reranker_base["EventDate"] >= history_cut_date)
    & (reranker_base["EventDate"] < dev_cut_date)
].copy()

reranker_dev_queries = reranker_base.loc[
    reranker_base["EventDate"] >= dev_cut_date
].copy()

print("Outer chronological split")
for name, frame in [
    ("Train", data_train),
    ("Validation", data_validation),
    ("Test", data_test),
]:
    print(
        f"{name:10s}: {len(frame):,} rows | "
        f"{frame['EventDate'].min().date()} -> {frame['EventDate'].max().date()} | "
        f"{frame['AnimalId'].nunique():,} AnimalIds"
    )

print("\nInternal reranker split (inside outer train only)")
for name, frame in [
    ("History", reranker_history),
    ("Train queries", reranker_train_queries),
    ("Dev queries", reranker_dev_queries),
]:
    print(
        f"{name:13s}: {len(frame):,} rows | "
        f"{frame['EventDate'].min().date()} -> {frame['EventDate'].max().date()}"
    )


Outer chronological split
Train     : 3,985,088 rows | 2019-06-14 -> 2020-12-13 | 7,539 AnimalIds
Validation: 1,327,822 rows | 2020-12-14 -> 2021-05-20 | 5,601 AnimalIds
Test      : 1,332,262 rows | 2021-05-21 -> 2021-10-24 | 5,648 AnimalIds

Internal reranker split (inside outer train only)
History      : 2,787,167 rows | 2019-06-14 -> 2020-07-05
Train queries: 592,716 rows | 2020-07-06 -> 2020-09-20
Dev queries  : 605,205 rows | 2020-09-21 -> 2020-12-13



## Final pipeline design

The executable pipeline below keeps only the components that survived the optimization process:

1. Build a leakage-safe reference snapshot.
2. Generate **Global Top-320** structural candidates for the same lactation number.
3. Compute local KNN-like distance features, cow/lactation prototypes, residual features, recency, and within-query ranks.
4. Restrict to **Top-160** candidates for the HGB reranker.
5. Train the final HGB on two complementary snapshot families:
   - **Fresh snapshots** for short/medium reference freshness.
   - **Long-gap snapshots** for 1–160 day reference aging.
6. Evaluate on a fixed 2,000-query chronological outer-validation sample.

No outer-test labels are used below.


In [4]:

# ============================================================
# 3. Structural, distance, and reference helper functions
# ============================================================

BASE_NUMERIC_FEATURES = [
    "DaysInMilk",
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
]


def event_day_from_origin(frame, origin_date):
    return (
        pd.to_datetime(frame["EventDate"], errors="coerce") - origin_date
    ).dt.days.astype("float32")


def lactation_start_day(frame, origin_date):
    event_date = pd.to_datetime(frame["EventDate"], errors="coerce")
    lactation_start = (
        event_date
        - pd.to_timedelta(frame["DaysInMilk"], unit="D")
    )
    return (lactation_start - origin_date).dt.days.astype("float32")


def build_structural_profiles(
    labelled_frame,
    class_to_number,
    origin_date,
    include_projection=True,
):
    temp = labelled_frame[
        ["AnimalId", "LactationNumber", "EventDate", "DaysInMilk"]
    ].copy()

    temp["EventDay"] = event_day_from_origin(temp, origin_date)
    temp["LactationStartDay"] = lactation_start_day(temp, origin_date)

    temp = temp.dropna(
        subset=[
            "AnimalId",
            "LactationNumber",
            "DaysInMilk",
            "EventDay",
            "LactationStartDay",
        ]
    )

    observed = (
        temp.groupby(["AnimalId", "LactationNumber"], observed=True)
        .agg(
            LactationStartDay=("LactationStartDay", "median"),
            LastEventDay=("EventDay", "max"),
            LastDaysInMilk=("DaysInMilk", "max"),
        )
        .reset_index()
    )

    observed["ClassId"] = (
        observed["AnimalId"].astype(object).map(class_to_number)
    )
    observed = observed.loc[observed["ClassId"].notna()].copy()

    observed["ClassId"] = observed["ClassId"].astype(np.int32)
    observed["LactationNumber"] = (
        observed["LactationNumber"].round().astype(np.int16)
    )

    for col in ["LactationStartDay", "LastEventDay", "LastDaysInMilk"]:
        observed[col] = observed[col].round().astype(np.int32)

    observed["IsProjected"] = False

    interval_rows = []
    for class_id, group in observed.groupby("ClassId", sort=False):
        group = group.sort_values("LactationNumber")
        lactations = group["LactationNumber"].to_numpy()
        starts = group["LactationStartDay"].to_numpy()

        for i in range(1, len(group)):
            if lactations[i] - lactations[i - 1] == 1:
                interval = starts[i] - starts[i - 1]
                if 250 <= interval <= 550:
                    interval_rows.append(
                        {
                            "ClassId": int(class_id),
                            "IntervalDays": int(interval),
                        }
                    )

    if interval_rows:
        interval_df = pd.DataFrame(interval_rows)
        global_interval = int(round(interval_df["IntervalDays"].median()))
        animal_interval = (
            interval_df.groupby("ClassId")["IntervalDays"].median().to_dict()
        )
    else:
        global_interval = 365
        animal_interval = {}

    if not include_projection:
        return observed, global_interval

    projected_rows = []

    for class_id, group in observed.groupby("ClassId", sort=False):
        last = group.sort_values("LactationNumber").iloc[-1]
        last_lactation = int(last["LactationNumber"])

        if last_lactation >= 10:
            continue

        interval = int(
            round(animal_interval.get(int(class_id), global_interval))
        )

        projected_rows.append(
            {
                "AnimalId": last["AnimalId"],
                "LactationNumber": last_lactation + 1,
                "LactationStartDay": int(last["LactationStartDay"] + interval),
                "LastEventDay": int(last["LastEventDay"]),
                "LastDaysInMilk": int(last["LastDaysInMilk"]),
                "ClassId": int(class_id),
                "IsProjected": True,
            }
        )

    if projected_rows:
        profiles = pd.concat(
            [observed, pd.DataFrame(projected_rows)],
            ignore_index=True,
        )
    else:
        profiles = observed.copy()

    return profiles, global_interval


def make_profile_lookup(profiles):
    lookup = {}

    for lactation, group in profiles.groupby("LactationNumber", sort=False):
        lookup[int(lactation)] = {
            "class_ids": group["ClassId"].to_numpy(dtype=np.int32),
            "start_days": group["LactationStartDay"].to_numpy(dtype=np.float32),
            "last_event_days": group["LastEventDay"].to_numpy(dtype=np.float32),
            "last_dim": group["LastDaysInMilk"].to_numpy(dtype=np.float32),
            "projected": group["IsProjected"].to_numpy(dtype=bool),
        }

    return lookup


def get_candidate_profiles(
    profile_lookup,
    lactation_number,
    query_start_day,
    max_candidates=GLOBAL_MAX_CANDIDATES,
    projected_rank_penalty_days=PROJECTED_RANK_PENALTY_DAYS,
):
    """Global structural retrieval: no hard start-date tolerance."""
    if pd.isna(lactation_number) or pd.isna(query_start_day):
        return None

    lactation_number = int(round(float(lactation_number)))
    profile = profile_lookup.get(lactation_number)

    if profile is None:
        return None

    start_difference = np.abs(
        profile["start_days"] - float(query_start_day)
    )

    structural_score = (
        start_difference
        + profile["projected"].astype(np.float32)
        * projected_rank_penalty_days
    )

    positions = np.argsort(structural_score)[
        : min(max_candidates, len(structural_score))
    ]

    return {
        key: value[positions]
        for key, value in profile.items()
    } | {
        "start_difference": start_difference[positions]
    }


def create_distance_features(raw_frame):
    X = raw_frame[
        BASE_NUMERIC_FEATURES + ["ReproductionStatus"]
    ].copy()

    duration_min = X["DurationSession_sec"].astype(np.float32) / 60.0

    X["YieldPerMinute"] = (
        X["YieldSession"] / duration_min.replace(0, np.nan)
    )
    X["First2MinFraction"] = (
        X["YieldFirst2Min_Session"]
        / X["YieldSession"].replace(0, np.nan)
    )
    X["FlowShapeRatio"] = (
        X["Flow30_60Session"]
        / X["Avgmilkflow"].replace(0, np.nan)
    )

    return X


def fit_distance_preprocessor(raw_frame):
    features = create_distance_features(raw_frame)

    numeric_columns = [
        c for c in features.columns
        if c != "ReproductionStatus"
    ]

    numeric = features[numeric_columns].copy()
    numeric_fill = numeric.median(numeric_only=True)
    numeric = numeric.fillna(numeric_fill)

    scaler = StandardScaler()
    numeric_scaled = scaler.fit_transform(numeric).astype(np.float32)

    reproduction = pd.get_dummies(
        features["ReproductionStatus"],
        prefix="ReproductionStatus",
        dtype=np.float32,
    )

    reproduction_columns = list(reproduction.columns)

    X = np.column_stack(
        [
            numeric_scaled,
            reproduction.to_numpy(dtype=np.float32),
        ]
    ).astype(np.float32)

    feature_names = numeric_columns + reproduction_columns

    return (
        X,
        numeric_columns,
        numeric_fill,
        scaler,
        reproduction_columns,
        feature_names,
    )


def transform_distance_features(
    raw_frame,
    numeric_columns,
    numeric_fill,
    scaler,
    reproduction_columns,
):
    features = create_distance_features(raw_frame)

    numeric = (
        features[numeric_columns]
        .copy()
        .fillna(numeric_fill)
    )

    numeric_scaled = scaler.transform(numeric).astype(np.float32)

    reproduction = (
        pd.get_dummies(
            features["ReproductionStatus"],
            prefix="ReproductionStatus",
            dtype=np.float32,
        )
        .reindex(columns=reproduction_columns, fill_value=0)
    )

    return np.column_stack(
        [
            numeric_scaled,
            reproduction.to_numpy(dtype=np.float32),
        ]
    ).astype(np.float32)


def sample_reference_rows(
    frame,
    max_rows_per_group=REFERENCE_ROWS_PER_GROUP,
    recent_fraction=0.80,
    random_state=RANDOM_STATE,
):
    selected_indices = []

    grouped = frame.groupby(
        ["AnimalId", "LactationNumber"],
        observed=True,
        sort=False,
    )

    for _, group in grouped:
        group = group.sort_values("EventDate")

        if len(group) <= max_rows_per_group:
            selected_indices.extend(group.index.tolist())
            continue

        n_recent = max(
            1,
            int(round(max_rows_per_group * recent_fraction)),
        )

        recent = group.tail(n_recent)
        remaining_n = max_rows_per_group - len(recent)

        if remaining_n > 0:
            historical_pool = group.loc[
                ~group.index.isin(recent.index)
            ]

            historical = historical_pool.sample(
                n=min(remaining_n, len(historical_pool)),
                replace=False,
                random_state=random_state,
            )

            selected = pd.concat([recent, historical])
        else:
            selected = recent

        selected_indices.extend(selected.index.tolist())

    return pd.Index(selected_indices)


def build_reference_indices(class_ids, lactations):
    """Build exact class/lactation, per-class lactation, and class-position indices."""
    exact = {}
    class_lactations = {}
    class_positions = {}

    for row_position, (class_id, lactation) in enumerate(
        zip(class_ids, lactations)
    ):
        class_id = int(class_id)
        lactation = int(lactation)

        exact.setdefault((class_id, lactation), []).append(row_position)
        class_lactations.setdefault(class_id, set()).add(lactation)
        class_positions.setdefault(class_id, []).append(row_position)

    exact = {
        key: np.asarray(positions, dtype=np.int32)
        for key, positions in exact.items()
    }

    class_lactations = {
        class_id: np.asarray(sorted(values), dtype=np.int16)
        for class_id, values in class_lactations.items()
    }

    class_positions = {
        class_id: np.asarray(positions, dtype=np.int32)
        for class_id, positions in class_positions.items()
    }

    return exact, class_lactations, class_positions


def choose_reference_lactation(
    class_id,
    query_lactation,
    class_lactations,
):
    available = class_lactations.get(int(class_id))

    if available is None or len(available) == 0:
        return None, 99

    query_lactation = int(round(float(query_lactation)))

    if query_lactation in available:
        return query_lactation, 0

    previous = available[available <= query_lactation]

    if len(previous) > 0:
        selected = int(previous.max())
    else:
        selected = int(
            available[np.argmin(np.abs(available - query_lactation))]
        )

    return selected, abs(query_lactation - selected)


def get_nearest_lactation_prototype(
    prototype_table,
    class_id,
    lactation_number,
    query_dim,
):
    prototype_dict = prototype_table.get(
        (int(class_id), int(lactation_number))
    )

    if not prototype_dict:
        return None

    query_bin = int(np.floor(float(query_dim) / DIM_BIN_WIDTH))
    available_bins = np.asarray(
        list(prototype_dict.keys()),
        dtype=np.int16,
    )

    nearest_bin = int(
        available_bins[
            np.argmin(np.abs(available_bins - query_bin))
        ]
    )

    return prototype_dict[nearest_bin]


print("Core helper functions loaded.")


Core helper functions loaded.


In [5]:

# ============================================================
# 4. Reference-system builder and candidate-feature pipeline
# ============================================================

RANKING_COLUMNS = [
    "StartDifference",
    "RecencyDays",
    "ClosestDIMDifference",
    "LocalMinManhattan",
    "LocalMean3Manhattan",
    "LocalMean5Manhattan",
    "LocalMinEuclidean",
    "LocalMean3Euclidean",
    "LactationPrototypeManhattan",
    "LactationPrototypeEuclidean",
    "CowPrototypeManhattan",
    "CowPrototypeEuclidean",
    "ReproductionDistance",
    "Residual_DaysInMilk",
    "Residual_Avgmilkflow",
    "Residual_Flow30_60Session",
    "Residual_YieldFirst2Min_Session",
    "Residual_YieldSession",
    "Residual_DurationSession_sec",
    "Residual_YieldPerMinute",
    "Residual_First2MinFraction",
    "Residual_FlowShapeRatio",
]

NON_FEATURE_COLUMNS = [
    "QueryId",
    "CandidateClassId",
    "TrueClassId",
    "IsCorrectCandidate",
]


def build_reference_system(reference_df):
    """Build all snapshot-specific objects in one dictionary.

    Keeping the state inside a dictionary avoids the fragile global-variable
    swapping used in the exploratory notebook.
    """
    class_values = pd.Index(
        reference_df["AnimalId"].astype(object).unique()
    )

    class_to_number = {
        animal_id: class_number
        for class_number, animal_id in enumerate(class_values)
    }

    origin_date = pd.to_datetime(reference_df["EventDate"]).min()

    profiles, global_lactation_interval = build_structural_profiles(
        reference_df,
        class_to_number,
        origin_date,
        include_projection=True,
    )

    profile_lookup = make_profile_lookup(profiles)

    reference_indices = sample_reference_rows(reference_df)

    reference_raw = reference_df.loc[reference_indices].copy()

    (
        X_reference,
        numeric_columns,
        numeric_fill,
        scaler,
        reproduction_columns,
        feature_names,
    ) = fit_distance_preprocessor(reference_raw)

    reference_class_ids = (
        reference_raw["AnimalId"]
        .astype(object)
        .map(class_to_number)
        .astype(np.int32)
        .to_numpy()
    )

    reference_lactations = (
        reference_raw["LactationNumber"]
        .round()
        .astype(np.int16)
        .to_numpy()
    )

    reference_dim = (
        reference_raw["DaysInMilk"]
        .astype(np.float32)
        .to_numpy()
    )

    (
        reference_exact_index,
        reference_class_lactations,
        class_positions,
    ) = build_reference_indices(
        reference_class_ids,
        reference_lactations,
    )

    # Whole-cow prototypes. class_positions avoids repeatedly scanning
    # the entire reference array for each class.
    cow_prototype = {
        class_id: np.median(
            X_reference[positions],
            axis=0,
        ).astype(np.float32)
        for class_id, positions in class_positions.items()
    }

    # Lactation + DIM-bin prototypes.
    reference_dim_bin = np.floor(
        reference_dim / DIM_BIN_WIDTH
    ).astype(np.int16)

    lactation_prototype = {}

    for (class_id, lactation_number), positions in (
        reference_exact_index.items()
    ):
        bins_here = reference_dim_bin[positions]
        prototype_by_bin = {}

        for dim_bin in np.unique(bins_here):
            bin_positions = positions[bins_here == dim_bin]

            prototype_by_bin[int(dim_bin)] = np.median(
                X_reference[bin_positions],
                axis=0,
            ).astype(np.float32)

        lactation_prototype[
            (int(class_id), int(lactation_number))
        ] = prototype_by_bin

    system = {
        "class_values": class_values,
        "class_to_number": class_to_number,
        "origin_date": origin_date,
        "global_lactation_interval": global_lactation_interval,
        "profile_lookup": profile_lookup,
        "X_reference": X_reference,
        "numeric_columns": numeric_columns,
        "numeric_fill": numeric_fill,
        "scaler": scaler,
        "reproduction_columns": reproduction_columns,
        "feature_names": feature_names,
        "reference_dim": reference_dim,
        "reference_exact_index": reference_exact_index,
        "reference_class_lactations": reference_class_lactations,
        "cow_prototype": cow_prototype,
        "lactation_prototype": lactation_prototype,
        "n_reference_rows": len(reference_raw),
        "n_profiles": len(profiles),
    }

    return system


def _nearest_mean(distances, k):
    k = min(k, len(distances))
    return float(
        np.mean(
            np.partition(distances, k - 1)[:k]
        )
    )


def build_candidate_features_for_one_query(
    query_frame,
    query_id,
    ref,
):
    if len(query_frame) != 1:
        raise ValueError("query_frame must contain exactly one row.")

    query_vector = transform_distance_features(
        query_frame,
        ref["numeric_columns"],
        ref["numeric_fill"],
        ref["scaler"],
        ref["reproduction_columns"],
    )[0]

    query_event_day = float(
        event_day_from_origin(
            query_frame,
            ref["origin_date"],
        ).iloc[0]
    )

    query_start_day = float(
        lactation_start_day(
            query_frame,
            ref["origin_date"],
        ).iloc[0]
    )

    query_dim = float(query_frame["DaysInMilk"].iloc[0])
    query_lactation = int(
        round(float(query_frame["LactationNumber"].iloc[0]))
    )

    true_animal_id = query_frame["AnimalId"].astype(object).iloc[0]
    true_class_id = ref["class_to_number"].get(
        true_animal_id,
        np.nan,
    )

    candidate = get_candidate_profiles(
        ref["profile_lookup"],
        query_lactation,
        query_start_day,
    )

    if candidate is None:
        return pd.DataFrame()

    candidate_count = len(candidate["class_ids"])

    numeric_feature_positions = {
        name: position
        for position, name in enumerate(ref["feature_names"])
        if not name.startswith("ReproductionStatus_")
    }

    reproduction_positions = [
        position
        for position, name in enumerate(ref["feature_names"])
        if name.startswith("ReproductionStatus_")
    ]

    rows = []

    for candidate_position in range(candidate_count):
        class_id = int(candidate["class_ids"][candidate_position])
        start_difference = float(
            candidate["start_difference"][candidate_position]
        )
        is_projected = int(
            candidate["projected"][candidate_position]
        )
        last_event_day = float(
            candidate["last_event_days"][candidate_position]
        )

        (
            selected_lactation,
            lactation_gap,
        ) = choose_reference_lactation(
            class_id,
            query_lactation,
            ref["reference_class_lactations"],
        )

        if selected_lactation is None:
            continue

        reference_positions = ref["reference_exact_index"].get(
            (class_id, selected_lactation)
        )

        if reference_positions is None or len(reference_positions) == 0:
            continue

        dim_difference = np.abs(
            ref["reference_dim"][reference_positions] - query_dim
        )

        closest_dim_difference = float(np.min(dim_difference))

        inside_dim = dim_difference <= DIM_TOLERANCE_DAYS

        if inside_dim.any():
            usable_positions = reference_positions[inside_dim]
        else:
            nearest_order = np.argsort(dim_difference)
            usable_positions = reference_positions[
                nearest_order[: min(10, len(nearest_order))]
            ]

        differences = (
            ref["X_reference"][usable_positions] - query_vector
        )

        manhattan_distances = np.sum(
            np.abs(differences),
            axis=1,
        )

        euclidean_distances = np.sqrt(
            np.sum(differences ** 2, axis=1)
        )

        local_min_manhattan = float(np.min(manhattan_distances))
        local_mean3_manhattan = _nearest_mean(manhattan_distances, 3)
        local_mean5_manhattan = _nearest_mean(manhattan_distances, 5)
        local_min_euclidean = float(np.min(euclidean_distances))
        local_mean3_euclidean = _nearest_mean(euclidean_distances, 3)

        lactation_prototype = get_nearest_lactation_prototype(
            ref["lactation_prototype"],
            class_id,
            selected_lactation,
            query_dim,
        )

        residual_features = {}

        if lactation_prototype is not None:
            prototype_difference = query_vector - lactation_prototype

            lactation_manhattan = float(
                np.sum(np.abs(prototype_difference))
            )

            lactation_euclidean = float(
                np.sqrt(np.sum(prototype_difference ** 2))
            )

            for feature_name, position in numeric_feature_positions.items():
                residual_features[
                    f"Residual_{feature_name}"
                ] = float(abs(prototype_difference[position]))

            if reproduction_positions:
                reproduction_distance = float(
                    np.sum(
                        np.abs(
                            prototype_difference[reproduction_positions]
                        )
                    )
                )
            else:
                reproduction_distance = 0.0

        else:
            lactation_manhattan = np.nan
            lactation_euclidean = np.nan
            reproduction_distance = np.nan

            for feature_name in numeric_feature_positions:
                residual_features[
                    f"Residual_{feature_name}"
                ] = np.nan

        cow_prototype = ref["cow_prototype"].get(class_id)

        if cow_prototype is not None:
            cow_difference = query_vector - cow_prototype

            cow_manhattan = float(
                np.sum(np.abs(cow_difference))
            )

            cow_euclidean = float(
                np.sqrt(np.sum(cow_difference ** 2))
            )
        else:
            cow_manhattan = np.nan
            cow_euclidean = np.nan

        recency_days = max(
            0.0,
            query_event_day - last_event_day,
        )

        row = {
            "QueryId": query_id,
            "CandidateClassId": class_id,
            "TrueClassId": true_class_id,
            "IsCorrectCandidate": (
                int(class_id == true_class_id)
                if not pd.isna(true_class_id)
                else 0
            ),
            "CandidateCount": candidate_count,
            "CandidateStartRank": candidate_position + 1,
            "StartDifference": start_difference,
            "IsProjected": is_projected,
            "RecencyDays": recency_days,
            "LogRecencyDays": np.log1p(recency_days),
            "LactationGap": lactation_gap,
            "QueryDaysInMilk": query_dim,
            "QueryLactationNumber": query_lactation,
            "ClosestDIMDifference": closest_dim_difference,
            "ReferenceRowCount": len(reference_positions),
            "LocalMinManhattan": local_min_manhattan,
            "LocalMean3Manhattan": local_mean3_manhattan,
            "LocalMean5Manhattan": local_mean5_manhattan,
            "LocalMinEuclidean": local_min_euclidean,
            "LocalMean3Euclidean": local_mean3_euclidean,
            "LactationPrototypeManhattan": lactation_manhattan,
            "LactationPrototypeEuclidean": lactation_euclidean,
            "CowPrototypeManhattan": cow_manhattan,
            "CowPrototypeEuclidean": cow_euclidean,
            "ReproductionDistance": reproduction_distance,
        }

        row.update(residual_features)
        rows.append(row)

    return pd.DataFrame(rows)


def add_candidate_rank_features(candidate_pairs):
    result = candidate_pairs.copy()

    for column in RANKING_COLUMNS:
        if column not in result.columns:
            continue

        result[f"{column}_Rank"] = (
            result.groupby("QueryId")[column]
            .rank(
                method="average",
                ascending=True,
                pct=True,
                na_option="bottom",
            )
            .astype(np.float32)
        )

    return result


def build_candidate_dataset(
    query_frame,
    prefix,
    ref,
    progress_every=100,
):
    """Build Global Top-320 candidate rows.

    Rank features are added later after Top-160 filtering, which avoids
    computing ranks twice in a clean run.
    """
    tables = []

    for counter, query_index in enumerate(
        query_frame.index,
        start=1,
    ):
        one_query = query_frame.loc[[query_index]].copy()

        candidate_table = build_candidate_features_for_one_query(
            one_query,
            query_id=f"{prefix}_{query_index}",
            ref=ref,
        )

        if len(candidate_table) > 0:
            tables.append(candidate_table)

        if (
            counter % progress_every == 0
            or counter == len(query_frame)
        ):
            print(
                f"{prefix}: processed "
                f"{counter} / {len(query_frame)} queries"
            )

    if not tables:
        return pd.DataFrame()

    return pd.concat(tables, ignore_index=True)


def prepare_topk_for_model(
    candidate_pairs,
    top_k=MODEL_TOP_K,
    require_positive=False,
):
    result = (
        candidate_pairs.loc[
            candidate_pairs["CandidateStartRank"] <= top_k
        ]
        .copy()
        .reset_index(drop=True)
    )

    result["CandidateCount"] = (
        result.groupby("QueryId")["QueryId"]
        .transform("size")
        .astype(np.int16)
    )

    # Recompute ranks inside the actual Top-K model candidate set.
    result = add_candidate_rank_features(result)

    if require_positive:
        positive_available = (
            result.groupby("QueryId")["IsCorrectCandidate"]
            .transform("max")
        )
        result = (
            result.loc[positive_available == 1]
            .copy()
            .reset_index(drop=True)
        )

    return result


def load_or_build_snapshot_candidates(
    reference_df,
    query_df,
    prefix,
    cache_path,
    progress_every=100,
):
    expected_ids = {
        f"{prefix}_{idx}"
        for idx in query_df.index
    }

    if cache_path.exists():
        cached = pd.read_pickle(cache_path)
        cached_ids = set(cached["QueryId"].unique())

        # Allow a tiny fraction of zero-candidate queries while rejecting
        # an obviously stale cache from a different query plan.
        if (
            cached_ids.issubset(expected_ids)
            and len(cached_ids) >= 0.95 * len(expected_ids)
        ):
            print("Loaded cache:", cache_path.name)
            return cached

        print("Cache/query mismatch; rebuilding:", cache_path.name)

    ref = build_reference_system(reference_df)

    print(
        f"Reference: {len(reference_df):,} rows | "
        f"{len(ref['class_values']):,} AnimalIds | "
        f"{ref['n_reference_rows']:,} sampled rows | "
        f"matrix {ref['X_reference'].shape}"
    )

    candidate_pairs = build_candidate_dataset(
        query_df,
        prefix=prefix,
        ref=ref,
        progress_every=progress_every,
    )

    candidate_pairs.to_pickle(cache_path)
    print("Saved cache:", cache_path.name)

    del ref
    gc.collect()

    return candidate_pairs


print("Reference and candidate pipeline loaded.")


Reference and candidate pipeline loaded.



## Fresh multi-snapshot training family

This family preserves the successful short/medium-horizon behavior discovered during optimization.  
Five historical reference cutoffs are used, and 2,000 query/snapshot pairs are sampled to match the internal development-set distribution of **days since the true cow was last seen in the same lactation**.


In [6]:

# ============================================================
# 5. Build the Fresh multi-snapshot query plan
# ============================================================

fresh_train_period = reranker_train_queries.copy()
fresh_train_period["EventDate"] = pd.to_datetime(
    fresh_train_period["EventDate"]
)

# The late 30% of the internal train-query period supplies candidate queries.
unique_train_dates = np.sort(
    fresh_train_period["EventDate"].dt.normalize().unique()
)

cutoff_position = int(len(unique_train_dates) * 0.70)
fresh_reference_cutoff = pd.Timestamp(
    unique_train_dates[cutoff_position - 1]
)

fresh_model_train_queries = (
    fresh_train_period.loc[
        fresh_train_period["EventDate"].dt.normalize()
        > fresh_reference_cutoff
    ]
    .copy()
)

# Candidate pool used to reproduce the DEV freshness profile.
snapshot_query_sample = (
    fresh_model_train_queries
    .sample(
        n=min(20_000, len(fresh_model_train_queries)),
        random_state=2026,
    )
    [["AnimalId", "LactationNumber", "EventDate"]]
    .copy()
)

snapshot_query_sample["OriginalQueryIndex"] = (
    snapshot_query_sample.index
)


def assign_freshness_bin(days):
    if pd.isna(days):
        return "MissingSameLactation"
    if days <= 14:
        return "00_0_14"
    if days <= 30:
        return "01_15_30"
    if days <= 60:
        return "02_31_60"
    if days <= 90:
        return "03_61_90"
    return "04_Over90"


snapshot_pair_tables = []

for snapshot_date in FRESH_SNAPSHOT_DATES:
    snapshot_reference = pd.concat(
        [
            reranker_history,
            fresh_train_period.loc[
                fresh_train_period["EventDate"] <= snapshot_date
            ],
        ],
        ignore_index=True,
    )

    snapshot_last_seen = (
        snapshot_reference[
            ["AnimalId", "LactationNumber", "EventDate"]
        ]
        .dropna()
        .groupby(
            ["AnimalId", "LactationNumber"],
            as_index=False,
        )["EventDate"]
        .max()
        .rename(columns={"EventDate": "LastReferenceDate"})
    )

    temp = snapshot_query_sample.merge(
        snapshot_last_seen,
        on=["AnimalId", "LactationNumber"],
        how="left",
    )

    temp = temp.loc[
        temp["EventDate"] > snapshot_date
    ].copy()

    temp["SnapshotDate"] = snapshot_date
    temp["DaysSinceLastSeen"] = (
        temp["EventDate"] - temp["LastReferenceDate"]
    ).dt.days
    temp["FreshnessBin"] = (
        temp["DaysSinceLastSeen"].apply(assign_freshness_bin)
    )

    snapshot_pair_tables.append(temp)


snapshot_training_pool = pd.concat(
    snapshot_pair_tables,
    ignore_index=True,
)

sampling_order = [
    "04_Over90",
    "00_0_14",
    "03_61_90",
    "MissingSameLactation",
    "01_15_30",
    "02_31_60",
]

selected_parts = []
used_original_queries = set()

for i, freshness_bin in enumerate(sampling_order):
    target_n = FRESH_TARGET_BIN_COUNTS[freshness_bin]

    available = (
        snapshot_training_pool.loc[
            (snapshot_training_pool["FreshnessBin"] == freshness_bin)
            & (
                ~snapshot_training_pool["OriginalQueryIndex"]
                .isin(used_original_queries)
            )
        ]
        .sample(frac=1, random_state=2026 + i)
        .drop_duplicates("OriginalQueryIndex", keep="first")
    )

    if len(available) < target_n:
        raise ValueError(
            f"Not enough unique queries for {freshness_bin}: "
            f"need {target_n}, have {len(available)}"
        )

    selected = available.sample(
        n=target_n,
        random_state=3000 + i,
    ).copy()

    selected_parts.append(selected)
    used_original_queries.update(
        selected["OriginalQueryIndex"].tolist()
    )


matched_snapshot_train = (
    pd.concat(selected_parts, ignore_index=True)
    .sample(frac=1, random_state=2026)
    .reset_index(drop=True)
)

assert len(matched_snapshot_train) == 2000
assert matched_snapshot_train["OriginalQueryIndex"].nunique() == 2000
assert (
    matched_snapshot_train["EventDate"]
    > matched_snapshot_train["SnapshotDate"]
).all()

fresh_metadata = matched_snapshot_train[
    [
        "OriginalQueryIndex",
        "SnapshotDate",
        "FreshnessBin",
        "DaysSinceLastSeen",
    ]
].copy()

matched_full_train_queries = (
    fresh_model_train_queries
    .loc[fresh_metadata["OriginalQueryIndex"].tolist()]
    .copy()
)

matched_full_train_queries["OriginalQueryIndex"] = (
    matched_full_train_queries.index
)

matched_full_train_queries = matched_full_train_queries.merge(
    fresh_metadata,
    on="OriginalQueryIndex",
    how="left",
    validate="one_to_one",
)

fresh_query_groups = {
    snapshot_date: matched_full_train_queries.loc[
        matched_full_train_queries["SnapshotDate"] == snapshot_date
    ].copy()
    for snapshot_date in FRESH_SNAPSHOT_DATES
}

fresh_plan_summary = (
    matched_snapshot_train["SnapshotDate"]
    .value_counts()
    .sort_index()
    .rename("Queries")
    .to_frame()
)

print("Fresh reference cutoff:", fresh_reference_cutoff.date())
print("Fresh training queries:", len(matched_snapshot_train))
display(fresh_plan_summary)


Fresh reference cutoff: 2020-08-27
Fresh training queries: 2000


,Queries
SnapshotDate,
2020-07-10,681
2020-07-20,344
2020-07-31,310
2020-08-10,222
2020-08-20,443


In [7]:

# ============================================================
# 6. Generate/load Fresh snapshot candidates and prepare Top-160
# ============================================================

fresh_candidate_tables = []

for snapshot_date in FRESH_SNAPSHOT_DATES:
    query_df = fresh_query_groups[snapshot_date]

    reference_df = pd.concat(
        [
            reranker_history,
            fresh_train_period.loc[
                fresh_train_period["EventDate"] <= snapshot_date
            ],
        ],
        ignore_index=True,
    )

    prefix = "fresh" + snapshot_date.strftime("%m%d") + "_query"
    cache_path = (
        CACHE_DIR
        / f"fresh_snapshot_candidates_{snapshot_date.strftime('%Y%m%d')}.pkl"
    )

    print("\nFresh snapshot:", snapshot_date.date())

    candidate_pairs = load_or_build_snapshot_candidates(
        reference_df=reference_df,
        query_df=query_df,
        prefix=prefix,
        cache_path=cache_path,
        progress_every=50,
    )

    fresh_candidate_tables.append(candidate_pairs)

    del candidate_pairs, reference_df
    gc.collect()


fresh_multisnapshot_candidates = pd.concat(
    fresh_candidate_tables,
    ignore_index=True,
)

del fresh_candidate_tables
gc.collect()

fresh_train160 = prepare_topk_for_model(
    fresh_multisnapshot_candidates,
    top_k=MODEL_TOP_K,
    require_positive=False,
)

fresh_presence160 = (
    fresh_train160.groupby("QueryId")["IsCorrectCandidate"].max()
)

fresh_train_coverage = fresh_presence160.sum() / 2000

fresh_train160_model = (
    fresh_train160.loc[
        fresh_train160.groupby("QueryId")["IsCorrectCandidate"]
        .transform("max")
        == 1
    ]
    .copy()
    .reset_index(drop=True)
)

print("\nFresh Top-160 training coverage:", f"{fresh_train_coverage:.2%}")
print(
    "Fresh model-training queries:",
    fresh_train160_model["QueryId"].nunique(),
)
print("Fresh model rows:", f"{len(fresh_train160_model):,}")

del fresh_multisnapshot_candidates, fresh_train160
gc.collect()



Fresh snapshot: 2020-07-10
Reference: 2,830,363 rows | 6,690 AnimalIds | 1,802,025 sampled rows | matrix (1802025, 13)
fresh0710_query: processed 50 / 681 queries
fresh0710_query: processed 100 / 681 queries
fresh0710_query: processed 150 / 681 queries
fresh0710_query: processed 200 / 681 queries
fresh0710_query: processed 250 / 681 queries
fresh0710_query: processed 300 / 681 queries
fresh0710_query: processed 350 / 681 queries
fresh0710_query: processed 400 / 681 queries
fresh0710_query: processed 450 / 681 queries
fresh0710_query: processed 500 / 681 queries
fresh0710_query: processed 550 / 681 queries
fresh0710_query: processed 600 / 681 queries
fresh0710_query: processed 650 / 681 queries
fresh0710_query: processed 681 / 681 queries
Saved cache: fresh_snapshot_candidates_20200710.pkl

Fresh snapshot: 2020-07-20
Reference: 2,914,658 rows | 6,746 AnimalIds | 1,838,191 sampled rows | matrix (1838191, 13)
fresh0720_query: processed 50 / 344 queries
fresh0720_query: processed 100 / 34

0


## Long-gap multi-snapshot training family

Outer validation revealed strong temporal drift: performance declined as the fixed training reference became 60–160 days old.  
The retained fix is a second 2,000-query training family whose reference ages explicitly span **1–160 days**, matching the outer-validation age distribution.


In [8]:

# ============================================================
# 7. Build the Long-gap multi-snapshot query plan
# ============================================================

rng = np.random.default_rng(RANDOM_STATE)
selected_rows = []
used_query_indices = set()

event_dates = pd.to_datetime(data_train["EventDate"])

for freshness_window, target_n in LONGGAP_TARGETS.items():
    low_day, high_day = LONGGAP_BOUNDS[freshness_window]

    base_n = target_n // len(LONGGAP_SNAPSHOT_DATES)
    remainder = target_n % len(LONGGAP_SNAPSHOT_DATES)

    snapshot_targets = [
        base_n + (1 if i < remainder else 0)
        for i in range(len(LONGGAP_SNAPSHOT_DATES))
    ]

    for snapshot_date, snapshot_target in zip(
        LONGGAP_SNAPSHOT_DATES,
        snapshot_targets,
    ):
        lag_days = (event_dates - snapshot_date).dt.days

        eligible_indices = data_train.index[
            lag_days.between(
                low_day,
                high_day,
                inclusive="both",
            )
            & (~data_train.index.isin(used_query_indices))
        ].to_numpy()

        if len(eligible_indices) < snapshot_target:
            raise ValueError(
                f"Not enough rows for {snapshot_date.date()} / "
                f"{freshness_window}: need {snapshot_target}, "
                f"have {len(eligible_indices)}"
            )

        sampled_indices = rng.choice(
            eligible_indices,
            size=snapshot_target,
            replace=False,
        )

        sampled = data_train.loc[sampled_indices].copy()
        sampled["OriginalQueryIndex"] = sampled.index
        sampled["SnapshotDate"] = snapshot_date
        sampled["DaysSinceSnapshot"] = (
            sampled["EventDate"] - snapshot_date
        ).dt.days
        sampled["FreshnessWindow"] = freshness_window

        selected_rows.append(sampled)
        used_query_indices.update(sampled_indices.tolist())


longgap_training_queries = pd.concat(
    selected_rows,
    ignore_index=False,
).copy()

assert len(longgap_training_queries) == 2000
assert (
    longgap_training_queries["OriginalQueryIndex"].nunique()
    == 2000
)
assert (
    longgap_training_queries["EventDate"]
    > longgap_training_queries["SnapshotDate"]
).all()
assert (
    longgap_training_queries["EventDate"].max()
    <= data_train["EventDate"].max()
)

longgap_bin_summary = (
    longgap_training_queries
    .groupby("FreshnessWindow", observed=True)
    .agg(
        Queries=("OriginalQueryIndex", "size"),
        MeanDays=("DaysSinceSnapshot", "mean"),
        MinDays=("DaysSinceSnapshot", "min"),
        MaxDays=("DaysSinceSnapshot", "max"),
    )
    .reindex(LONGGAP_TARGETS.keys())
)

longgap_snapshot_summary = (
    longgap_training_queries
    .groupby("SnapshotDate")
    .agg(
        Queries=("OriginalQueryIndex", "size"),
        MinQueryDate=("EventDate", "min"),
        MaxQueryDate=("EventDate", "max"),
    )
)

print("Long-gap training queries:", len(longgap_training_queries))
display(longgap_bin_summary)
display(longgap_snapshot_summary)


Long-gap training queries: 2000


,Queries,MeanDays,MinDays,MaxDays
FreshnessWindow,,,,
1-30 days,394,16.187817,1,30
31-60 days,375,45.709333,31,60
61-90 days,384,75.604167,61,90
91-120 days,361,106.576177,91,120
121-160 days,486,140.368313,121,160


,Queries,MinQueryDate,MaxQueryDate
SnapshotDate,,,
2020-04-01,502,2020-04-03,2020-09-08
2020-05-01,501,2020-05-02,2020-09-30
2020-06-01,499,2020-06-02,2020-11-08
2020-07-01,498,2020-07-03,2020-12-08


In [9]:

# ============================================================
# 8. Generate/load Long-gap snapshot candidates and prepare Top-160
# ============================================================

longgap_candidate_tables = []

for snapshot_date in LONGGAP_SNAPSHOT_DATES:
    query_df = longgap_training_queries.loc[
        longgap_training_queries["SnapshotDate"] == snapshot_date
    ].copy()

    reference_df = data_train.loc[
        data_train["EventDate"] <= snapshot_date
    ].copy()

    prefix = "longgap" + snapshot_date.strftime("%m%d") + "_query"
    cache_path = (
        CACHE_DIR
        / f"longgap_candidates_{snapshot_date.strftime('%Y%m%d')}_global320.pkl"
    )

    print("\nLong-gap snapshot:", snapshot_date.date())

    candidate_pairs = load_or_build_snapshot_candidates(
        reference_df=reference_df,
        query_df=query_df,
        prefix=prefix,
        cache_path=cache_path,
        progress_every=50,
    )

    longgap_candidate_tables.append(candidate_pairs)

    del candidate_pairs, reference_df, query_df
    gc.collect()


longgap_multisnapshot_candidates = pd.concat(
    longgap_candidate_tables,
    ignore_index=True,
)

del longgap_candidate_tables
gc.collect()

longgap_train160 = prepare_topk_for_model(
    longgap_multisnapshot_candidates,
    top_k=MODEL_TOP_K,
    require_positive=False,
)

longgap_presence160 = (
    longgap_train160.groupby("QueryId")["IsCorrectCandidate"].max()
)

longgap_train_coverage = longgap_presence160.sum() / 2000

longgap_train160_model = (
    longgap_train160.loc[
        longgap_train160.groupby("QueryId")["IsCorrectCandidate"]
        .transform("max")
        == 1
    ]
    .copy()
    .reset_index(drop=True)
)

print("\nLong-gap Top-160 training coverage:", f"{longgap_train_coverage:.2%}")
print(
    "Long-gap model-training queries:",
    longgap_train160_model["QueryId"].nunique(),
)
print("Long-gap model rows:", f"{len(longgap_train160_model):,}")

del longgap_multisnapshot_candidates, longgap_train160
gc.collect()



Long-gap snapshot: 2020-04-01
Reference: 2,015,148 rows | 6,118 AnimalIds | 1,441,801 sampled rows | matrix (1441801, 13)
longgap0401_query: processed 50 / 502 queries
longgap0401_query: processed 100 / 502 queries
longgap0401_query: processed 150 / 502 queries
longgap0401_query: processed 200 / 502 queries
longgap0401_query: processed 250 / 502 queries
longgap0401_query: processed 300 / 502 queries
longgap0401_query: processed 350 / 502 queries
longgap0401_query: processed 400 / 502 queries
longgap0401_query: processed 450 / 502 queries
longgap0401_query: processed 500 / 502 queries
longgap0401_query: processed 502 / 502 queries
Saved cache: longgap_candidates_20200401_global320.pkl

Long-gap snapshot: 2020-05-01
Reference: 2,257,505 rows | 6,285 AnimalIds | 1,552,679 sampled rows | matrix (1552679, 13)
longgap0501_query: processed 50 / 501 queries
longgap0501_query: processed 100 / 501 queries
longgap0501_query: processed 150 / 501 queries
longgap0501_query: processed 200 / 501 quer

0

In [10]:

# ============================================================
# 9. Train the final 52-feature Hybrid HGB
# ============================================================

FINAL_FEATURES = [
    column
    for column in fresh_train160_model.columns
    if column not in NON_FEATURE_COLUMNS
]

assert len(FINAL_FEATURES) == 52, (
    f"Expected 52 final features, found {len(FINAL_FEATURES)}"
)

assert set(FINAL_FEATURES) == {
    column
    for column in longgap_train160_model.columns
    if column not in NON_FEATURE_COLUMNS
}

HYBRID_COLUMNS = NON_FEATURE_COLUMNS + FINAL_FEATURES

hybrid_train160_model = pd.concat(
    [
        fresh_train160_model[HYBRID_COLUMNS],
        longgap_train160_model[HYBRID_COLUMNS],
    ],
    ignore_index=True,
)

X_hybrid_train = (
    hybrid_train160_model[FINAL_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

y_hybrid_train = (
    hybrid_train160_model["IsCorrectCandidate"]
    .astype(np.int8)
)

hybrid_hgb160 = HistGradientBoostingClassifier(
    **HGB_PARAMS
)

hybrid_hgb160.fit(
    X_hybrid_train,
    y_hybrid_train,
)

print("Final Hybrid HGB trained.")
print("Training queries:", hybrid_train160_model["QueryId"].nunique())
print("Training candidate rows:", f"{len(hybrid_train160_model):,}")
print("Positive rows:", int(y_hybrid_train.sum()))
print("Features:", len(FINAL_FEATURES))
print("Iterations used:", hybrid_hgb160.n_iter_)


Final Hybrid HGB trained.
Training queries: 3377
Training candidate rows: 539,751
Positive rows: 3377
Features: 52
Iterations used: 400



## Outer validation

The final model is evaluated on 2,000 queries sampled evenly across the entire outer-validation period:

- Validation period: **2020-12-14 to 2021-05-20**
- Reference data: **outer train only**
- Reference cutoff: **2020-12-13**
- Candidate set used by the model: **Top 160**

This section does not use the outer test split.


In [11]:

# ============================================================
# 10. Build/load the 2,000-query outer-validation candidate set
# ============================================================

validation_sorted = data_validation.sort_values("EventDate")

validation_positions = (
    np.linspace(
        0,
        len(validation_sorted) - 1,
        OUTER_VALIDATION_SAMPLE_SIZE,
    )
    .round()
    .astype(int)
)

outer_validation_queries = (
    validation_sorted
    .iloc[validation_positions]
    .copy()
)

assert len(outer_validation_queries) == OUTER_VALIDATION_SAMPLE_SIZE
assert (
    outer_validation_queries["EventDate"].min()
    > data_train["EventDate"].max()
)

OUTER_VAL_CUTOFF = data_train["EventDate"].max()
OUTER_VAL_PREFIX = "outerval_query"
OUTER_VAL_CACHE_PATH = (
    CACHE_DIR
    / "outer_validation_candidates_2000_global320.pkl"
)

outer_val_candidate_pairs = load_or_build_snapshot_candidates(
    reference_df=data_train,
    query_df=outer_validation_queries,
    prefix=OUTER_VAL_PREFIX,
    cache_path=OUTER_VAL_CACHE_PATH,
    progress_every=100,
)

outer_val160_ranked = prepare_topk_for_model(
    outer_val_candidate_pairs,
    top_k=MODEL_TOP_K,
    require_positive=False,
)

outer_val_presence = (
    outer_val160_ranked
    .groupby("QueryId")["IsCorrectCandidate"]
    .max()
)

outer_val_eval = outer_validation_queries[
    ["AnimalId", "EventDate"]
].copy()

outer_val_eval["QueryId"] = [
    f"{OUTER_VAL_PREFIX}_{idx}"
    for idx in outer_val_eval.index
]

train_animal_ids = set(
    data_train["AnimalId"].dropna().astype(object)
)

outer_val_eval["AnimalSeenInTrain"] = (
    outer_val_eval["AnimalId"]
    .astype(object)
    .isin(train_animal_ids)
)

outer_val_eval["CorrectCandidatePresent"] = (
    outer_val_eval["QueryId"]
    .map(outer_val_presence)
    .fillna(0)
    .astype(bool)
)

print("Outer validation queries:", len(outer_val_eval))
print(
    "AnimalId seen in TRAIN:",
    f"{outer_val_eval['AnimalSeenInTrain'].mean():.2%}",
)
print(
    "Top-160 candidate coverage:",
    f"{outer_val_eval['CorrectCandidatePresent'].mean():.2%}",
)


Reference: 3,985,088 rows | 7,539 AnimalIds | 2,323,643 sampled rows | matrix (2323643, 13)
outerval_query: processed 100 / 2000 queries
outerval_query: processed 200 / 2000 queries
outerval_query: processed 300 / 2000 queries
outerval_query: processed 400 / 2000 queries
outerval_query: processed 500 / 2000 queries
outerval_query: processed 600 / 2000 queries
outerval_query: processed 700 / 2000 queries
outerval_query: processed 800 / 2000 queries
outerval_query: processed 900 / 2000 queries
outerval_query: processed 1000 / 2000 queries
outerval_query: processed 1100 / 2000 queries
outerval_query: processed 1200 / 2000 queries
outerval_query: processed 1300 / 2000 queries
outerval_query: processed 1400 / 2000 queries
outerval_query: processed 1500 / 2000 queries
outerval_query: processed 1600 / 2000 queries
outerval_query: processed 1700 / 2000 queries
outerval_query: processed 1800 / 2000 queries
outerval_query: processed 1900 / 2000 queries
outerval_query: processed 2000 / 2000 queri

In [12]:

# ============================================================
# 11. Evaluate the final Hybrid HGB on outer validation
# ============================================================

missing_features = [
    feature
    for feature in FINAL_FEATURES
    if feature not in outer_val160_ranked.columns
]

assert not missing_features, (
    f"Missing outer-validation features: {missing_features}"
)

X_outer = (
    outer_val160_ranked[FINAL_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .astype(np.float32)
)

outer_probability = (
    hybrid_hgb160.predict_proba(X_outer)[:, 1]
)

outer_scored = outer_val160_ranked[
    [
        "QueryId",
        "CandidateClassId",
        "TrueClassId",
        "IsCorrectCandidate",
    ]
].copy()

outer_scored["Probability"] = outer_probability

outer_best = (
    outer_scored
    .sort_values(
        ["QueryId", "Probability"],
        ascending=[True, False],
    )
    .groupby("QueryId", as_index=False)
    .first()
)

outer_best["PredictionCorrect"] = (
    outer_best["CandidateClassId"]
    == outer_best["TrueClassId"]
)

hybrid_outer_eval = outer_val_eval.merge(
    outer_best[
        ["QueryId", "PredictionCorrect"]
    ],
    on="QueryId",
    how="left",
)

hybrid_outer_eval["PredictionCorrect"] = (
    hybrid_outer_eval["PredictionCorrect"]
    .fillna(False)
    .astype(bool)
)

outer_candidate_coverage = (
    hybrid_outer_eval["CorrectCandidatePresent"].mean()
)

outer_overall_accuracy = (
    hybrid_outer_eval["PredictionCorrect"].mean()
)

outer_seen_accuracy = (
    hybrid_outer_eval.loc[
        hybrid_outer_eval["AnimalSeenInTrain"],
        "PredictionCorrect",
    ].mean()
)

outer_conditional_accuracy = (
    hybrid_outer_eval.loc[
        hybrid_outer_eval["CorrectCandidatePresent"],
        "PredictionCorrect",
    ].mean()
)

outer_correct = int(
    hybrid_outer_eval["PredictionCorrect"].sum()
)

print("=" * 72)
print("FINAL OUTER VALIDATION — HYBRID HGB")
print("=" * 72)
print("Top-160 candidate coverage:", f"{outer_candidate_coverage:.2%}")
print("Overall accuracy:", f"{outer_overall_accuracy:.2%}")
print("Seen-AnimalId accuracy:", f"{outer_seen_accuracy:.2%}")
print(
    "Conditional ranking accuracy:",
    f"{outer_conditional_accuracy:.2%}",
)
print(
    "Correct predictions:",
    outer_correct,
    "/",
    len(hybrid_outer_eval),
)


FINAL OUTER VALIDATION — HYBRID HGB
Top-160 candidate coverage: 82.05%
Overall accuracy: 41.65%
Seen-AnimalId accuracy: 45.17%
Conditional ranking accuracy: 50.76%
Correct predictions: 833 / 2000


In [13]:

# ============================================================
# 12. Compact temporal-drift diagnostic
# ============================================================

drift_eval = hybrid_outer_eval.copy()

drift_eval["DaysSinceReferenceCutoff"] = (
    pd.to_datetime(drift_eval["EventDate"])
    - OUTER_VAL_CUTOFF
).dt.days

drift_eval["FreshnessWindow"] = pd.cut(
    drift_eval["DaysSinceReferenceCutoff"],
    bins=[0, 30, 60, 90, 120, 160],
    labels=[
        "1-30 days",
        "31-60 days",
        "61-90 days",
        "91-120 days",
        "121-160 days",
    ],
    include_lowest=True,
)

drift_rows = []

for window, subset in drift_eval.groupby(
    "FreshnessWindow",
    observed=True,
):
    candidate_present = subset.loc[
        subset["CorrectCandidatePresent"]
    ]

    drift_rows.append(
        {
            "FreshnessWindow": str(window),
            "Queries": len(subset),
            "CandidateCoverage": (
                subset["CorrectCandidatePresent"].mean() * 100
            ),
            "ConditionalAccuracy": (
                candidate_present["PredictionCorrect"].mean() * 100
                if len(candidate_present)
                else np.nan
            ),
            "OverallAccuracy": (
                subset["PredictionCorrect"].mean() * 100
            ),
        }
    )

outer_drift_summary = pd.DataFrame(drift_rows)
display(outer_drift_summary)


,FreshnessWindow,Queries,CandidateCoverage,ConditionalAccuracy,OverallAccuracy
0,1-30 days,394,96.192893,74.142480,71.319797
1,31-60 days,375,89.600000,55.059524,49.333333
2,61-90 days,384,84.375000,42.592593,35.937500
3,91-120 days,361,77.285319,36.559140,28.254848
4,121-160 days,486,66.460905,39.318885,26.131687


In [14]:

# ============================================================
# 13. Save the retained final model
# ============================================================

FINAL_MODEL_PATH = (
    CACHE_DIR / "hybrid_hgb_top160_final.joblib"
)
FINAL_FEATURES_PATH = (
    CACHE_DIR / "hybrid_hgb_top160_features.json"
)
FINAL_METADATA_PATH = (
    CACHE_DIR / "hybrid_hgb_top160_metadata.json"
)

joblib.dump(
    hybrid_hgb160,
    FINAL_MODEL_PATH,
)

with open(
    FINAL_FEATURES_PATH,
    "w",
) as f:
    json.dump(
        FINAL_FEATURES,
        f,
        indent=2,
    )

final_metadata = {
    "model": "HistGradientBoostingClassifier",
    "candidate_generation": "Global structural Top-320",
    "reranker_top_k": MODEL_TOP_K,
    "feature_count": len(FINAL_FEATURES),
    "training_design": "Fresh multi-snapshot + long-gap multi-snapshot",
    "hgb_parameters": HGB_PARAMS,
    "iterations_used": int(hybrid_hgb160.n_iter_),
    "fresh_top160_training_coverage": float(fresh_train_coverage),
    "longgap_top160_training_coverage": float(longgap_train_coverage),
    "outer_validation_queries": int(len(hybrid_outer_eval)),
    "outer_validation_reference_cutoff": str(OUTER_VAL_CUTOFF.date()),
    "outer_validation_candidate_coverage": float(outer_candidate_coverage),
    "outer_validation_overall_accuracy": float(outer_overall_accuracy),
    "outer_validation_seen_accuracy": float(outer_seen_accuracy),
    "outer_validation_conditional_accuracy": float(
        outer_conditional_accuracy
    ),
    "outer_validation_correct_predictions": int(outer_correct),
    "outer_test_used": False,
}

with open(
    FINAL_METADATA_PATH,
    "w",
) as f:
    json.dump(
        final_metadata,
        f,
        indent=2,
    )

print("Saved final model:", FINAL_MODEL_PATH)
print("Saved feature order:", FINAL_FEATURES_PATH)
print("Saved metadata:", FINAL_METADATA_PATH)


Saved final model: xgboost_cache/hybrid_hgb_top160_final.joblib
Saved feature order: xgboost_cache/hybrid_hgb_top160_features.json
Saved metadata: xgboost_cache/hybrid_hgb_top160_metadata.json



## Retained result from the optimization session

The final retained configuration is the **52-feature Hybrid HGB**. In the completed optimization run, the 2,000-query outer-validation sample produced:

- Top-160 candidate coverage: **82.05%**
- Overall accuracy: **41.65%**
- Seen-AnimalId accuracy: **45.17%**
- Conditional ranking accuracy: **50.76%**
- Correct predictions: **833 / 2,000**

The internal short-horizon confirmation result was higher (**53.90% overall**) but did not generalize to the longer outer-validation horizon because of reference aging. The long-gap snapshot family improved this temporal robustness, and combining fresh + long-gap training produced the best retained outer-validation result.

The outer test split remains untouched and should be reserved for final evaluation after the modeling protocol is frozen.


In [15]:
# ============================================================
# FINAL TEST — STEP 1
#
# Reference:
#   TRAIN + VALIDATION
#   through 2021-05-20
#
# Test:
#   2,000 evenly distributed queries
#
# IMPORTANT:
#   The Hybrid HGB is NOT retrained.
# ============================================================

FINAL_TEST_SAMPLE_SIZE = 2000


# ============================================================
# 1. Build TEST reference
# ============================================================

test_reference = (
    pd.concat(
        [
            data_train,
            data_validation,
        ],
        axis=0,
    )
    .sort_values("EventDate")
    .copy()
)


TEST_REFERENCE_CUTOFF = (
    test_reference["EventDate"].max()
)


# ============================================================
# 2. Leakage safety checks
# ============================================================

assert (
    TEST_REFERENCE_CUTOFF
    ==
    pd.Timestamp("2021-05-20")
), TEST_REFERENCE_CUTOFF


assert (
    data_test["EventDate"].min()
    >
    TEST_REFERENCE_CUTOFF
)


assert (
    test_reference["EventDate"].max()
    <
    data_test["EventDate"].min()
)


# ============================================================
# 3. Uniformly sample 2,000 TEST queries
# ============================================================

test_sorted = (
    data_test
    .sort_values("EventDate")
)


final_test_positions = (
    np.linspace(
        0,
        len(test_sorted) - 1,
        FINAL_TEST_SAMPLE_SIZE,
    )
    .round()
    .astype(int)
)


final_test_queries = (
    test_sorted
    .iloc[
        final_test_positions
    ]
    .copy()
)


# ============================================================
# 4. Safety checks
# ============================================================

assert (
    len(final_test_queries)
    ==
    FINAL_TEST_SAMPLE_SIZE
)


assert (
    final_test_queries.index.nunique()
    ==
    FINAL_TEST_SAMPLE_SIZE
)


assert (
    final_test_queries["EventDate"].min()
    >
    TEST_REFERENCE_CUTOFF
)


# ============================================================
# 5. Seen-AnimalId ceiling
# ============================================================

test_reference_animal_ids = set(
    test_reference[
        "AnimalId"
    ]
    .dropna()
    .astype(object)
)


final_test_seen_rate = (
    final_test_queries[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        test_reference_animal_ids
    )
    .mean()
)


# ============================================================
# 6. Diagnostics
# ============================================================

print("=" * 72)

print(
    "FINAL TEST SETUP"
)

print("=" * 72)

print(
    "TEST reference rows:",
    f"{len(test_reference):,}"
)

print(
    "Reference date range:",
    test_reference["EventDate"].min(),
    "→",
    test_reference["EventDate"].max()
)

print(
    "Reference AnimalIds:",
    f"{test_reference['AnimalId'].nunique():,}"
)

print()

print(
    "Final TEST queries:",
    len(final_test_queries)
)

print(
    "TEST date range:",
    final_test_queries["EventDate"].min(),
    "→",
    final_test_queries["EventDate"].max()
)

print(
    "AnimalId seen in TEST reference:",
    f"{final_test_seen_rate:.2%}"
)

print(
    "Unique AnimalIds in TEST sample:",
    final_test_queries[
        "AnimalId"
    ].nunique()
)

FINAL TEST SETUP
TEST reference rows: 5,312,910
Reference date range: 2019-06-14 00:00:00 → 2021-05-20 00:00:00
Reference AnimalIds: 8,311

Final TEST queries: 2000
TEST date range: 2021-05-21 00:00:00 → 2021-10-24 00:00:00
AnimalId seen in TEST reference: 90.60%
Unique AnimalIds in TEST sample: 1615


In [16]:
# ============================================================
# FINAL TEST — STEP 2
#
# Build Global Top-320 candidates using
# TRAIN + VALIDATION as historical reference.
# ============================================================

FINAL_TEST_PREFIX = "finaltest_query"

FINAL_TEST_CACHE_PATH = (
    CACHE_DIR
    /
    "final_test_candidates_2000_train_plus_validation_global320.pkl"
)


final_test_candidate_pairs = (
    load_or_build_snapshot_candidates(

        reference_df=
            test_reference,

        query_df=
            final_test_queries,

        prefix=
            FINAL_TEST_PREFIX,

        cache_path=
            FINAL_TEST_CACHE_PATH,

        progress_every=100,
    )
)


print()

print(
    "Global Top-320 candidate rows:",
    f"{len(final_test_candidate_pairs):,}"
)

print(
    "Queries with candidate rows:",
    final_test_candidate_pairs[
        "QueryId"
    ].nunique()
)

Reference: 5,312,910 rows | 8,311 AnimalIds | 2,869,893 sampled rows | matrix (2869893, 13)
finaltest_query: processed 100 / 2000 queries
finaltest_query: processed 200 / 2000 queries
finaltest_query: processed 300 / 2000 queries
finaltest_query: processed 400 / 2000 queries
finaltest_query: processed 500 / 2000 queries
finaltest_query: processed 600 / 2000 queries
finaltest_query: processed 700 / 2000 queries
finaltest_query: processed 800 / 2000 queries
finaltest_query: processed 900 / 2000 queries
finaltest_query: processed 1000 / 2000 queries
finaltest_query: processed 1100 / 2000 queries
finaltest_query: processed 1200 / 2000 queries
finaltest_query: processed 1300 / 2000 queries
finaltest_query: processed 1400 / 2000 queries
finaltest_query: processed 1500 / 2000 queries
finaltest_query: processed 1600 / 2000 queries
finaltest_query: processed 1700 / 2000 queries
finaltest_query: processed 1800 / 2000 queries
finaltest_query: processed 1900 / 2000 queries
finaltest_query: process

In [17]:
# ============================================================
# FINAL TEST — STEP 3
# Prepare exact Top-160 candidate set
# ============================================================

final_test160_ranked = (
    prepare_topk_for_model(

        final_test_candidate_pairs,

        top_k=MODEL_TOP_K,

        require_positive=False,
    )
)


# ============================================================
# Candidate coverage
# ============================================================

final_test_presence = (
    final_test160_ranked
    .groupby("QueryId")[
        "IsCorrectCandidate"
    ]
    .max()
)


final_test_eval = (
    final_test_queries[
        [
            "AnimalId",
            "EventDate",
        ]
    ]
    .copy()
)


final_test_eval[
    "QueryId"
] = [
    f"{FINAL_TEST_PREFIX}_{idx}"
    for idx in final_test_eval.index
]


final_test_eval[
    "AnimalSeenInReference"
] = (
    final_test_eval[
        "AnimalId"
    ]
    .astype(object)
    .isin(
        test_reference_animal_ids
    )
)


final_test_eval[
    "CorrectCandidatePresent"
] = (
    final_test_eval[
        "QueryId"
    ]
    .map(
        final_test_presence
    )
    .fillna(0)
    .astype(bool)
)


print("=" * 72)

print(
    "FINAL TEST — RETRIEVAL"
)

print("=" * 72)

print(
    "AnimalId seen rate:",
    f"{final_test_eval['AnimalSeenInReference'].mean():.2%}"
)

print(
    "Top-160 candidate coverage:",
    f"{final_test_eval['CorrectCandidatePresent'].mean():.2%}"
)

print(
    "Correct candidate present:",
    int(
        final_test_eval[
            "CorrectCandidatePresent"
        ].sum()
    ),
    "/",
    len(final_test_eval)
)

print(
    "Mean candidates/query:",
    f"{final_test160_ranked.groupby('QueryId').size().mean():.2f}"
)

FINAL TEST — RETRIEVAL
AnimalId seen rate: 90.60%
Top-160 candidate coverage: 78.30%
Correct candidate present: 1566 / 2000
Mean candidates/query: 159.94


In [18]:
# ============================================================
# FINAL TEST — STEP 4
#
# Load the FROZEN model from disk.
#
# NO retraining.
# NO tuning.
# ============================================================

FINAL_MODEL_PATH = (
    CACHE_DIR
    /
    "hybrid_hgb_top160_final.joblib"
)

FINAL_FEATURES_PATH = (
    CACHE_DIR
    /
    "hybrid_hgb_top160_features.json"
)


frozen_final_model = (
    joblib.load(
        FINAL_MODEL_PATH
    )
)


with open(
    FINAL_FEATURES_PATH,
    "r",
) as f:

    frozen_final_features = (
        json.load(f)
    )


assert (
    len(frozen_final_features)
    ==
    52
)


missing_test_features = [
    feature
    for feature
    in frozen_final_features

    if feature
    not in final_test160_ranked.columns
]


assert not missing_test_features, (
    f"Missing TEST features: "
    f"{missing_test_features}"
)


print(
    "Frozen Hybrid HGB loaded."
)

print(
    "Frozen feature count:",
    len(frozen_final_features)
)

Frozen Hybrid HGB loaded.
Frozen feature count: 52


In [19]:
# ============================================================
# FINAL TEST — STEP 5
# Frozen Hybrid HGB prediction
# ============================================================

X_final_test = (
    final_test160_ranked[
        frozen_final_features
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .astype(np.float32)
)


final_test_probability = (
    frozen_final_model
    .predict_proba(
        X_final_test
    )[:, 1]
)


final_test_scored = (
    final_test160_ranked[
        [
            "QueryId",
            "CandidateClassId",
            "TrueClassId",
            "IsCorrectCandidate",
        ]
    ]
    .copy()
)


final_test_scored[
    "Probability"
] = final_test_probability


# ============================================================
# Select highest-probability candidate per query
# ============================================================

final_test_best = (
    final_test_scored
    .sort_values(
        [
            "QueryId",
            "Probability",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "QueryId",
        as_index=False,
    )
    .first()
)


final_test_best[
    "PredictionCorrect"
] = (
    final_test_best[
        "CandidateClassId"
    ]
    ==
    final_test_best[
        "TrueClassId"
    ]
)


# ============================================================
# Merge back to query-level table
# ============================================================

final_test_eval = (
    final_test_eval
    .merge(
        final_test_best[
            [
                "QueryId",
                "PredictionCorrect",
            ]
        ],
        on="QueryId",
        how="left",
    )
)


final_test_eval[
    "PredictionCorrect"
] = (
    final_test_eval[
        "PredictionCorrect"
    ]
    .fillna(False)
    .astype(bool)
)

In [20]:
# ============================================================
# FINAL TEST — FINAL METRICS
# ============================================================

final_test_candidate_coverage = (
    final_test_eval[
        "CorrectCandidatePresent"
    ].mean()
)


final_test_overall_accuracy = (
    final_test_eval[
        "PredictionCorrect"
    ].mean()
)


final_test_seen_accuracy = (
    final_test_eval.loc[
        final_test_eval[
            "AnimalSeenInReference"
        ],
        "PredictionCorrect",
    ]
    .mean()
)


final_test_conditional_accuracy = (
    final_test_eval.loc[
        final_test_eval[
            "CorrectCandidatePresent"
        ],
        "PredictionCorrect",
    ]
    .mean()
)


final_test_correct = int(
    final_test_eval[
        "PredictionCorrect"
    ].sum()
)


print("\n" + "=" * 72)

print(
    "FINAL TEST — FROZEN HYBRID HGB"
)

print("=" * 72)

print(
    "Reference cutoff:",
    TEST_REFERENCE_CUTOFF.date()
)

print(
    "TEST queries:",
    len(final_test_eval)
)

print(
    "AnimalId seen in reference:",
    f"{final_test_eval['AnimalSeenInReference'].mean():.2%}"
)

print(
    "Top-160 candidate coverage:",
    f"{final_test_candidate_coverage:.2%}"
)

print(
    "Overall TEST accuracy:",
    f"{final_test_overall_accuracy:.2%}"
)

print(
    "Seen-AnimalId TEST accuracy:",
    f"{final_test_seen_accuracy:.2%}"
)

print(
    "Conditional ranking accuracy:",
    f"{final_test_conditional_accuracy:.2%}"
)

print(
    "Correct predictions:",
    final_test_correct,
    "/",
    len(final_test_eval)
)


FINAL TEST — FROZEN HYBRID HGB
Reference cutoff: 2021-05-20
TEST queries: 2000
AnimalId seen in reference: 90.60%
Top-160 candidate coverage: 78.30%
Overall TEST accuracy: 43.55%
Seen-AnimalId TEST accuracy: 48.07%
Conditional ranking accuracy: 55.62%
Correct predictions: 871 / 2000


## Next step: Prediction

The frozen Hybrid HGB model is now ready for inference.

In this next step, we will:
- load the saved model and feature order from disk,
- build the Top-320 candidate set for the unseen query set,
- restrict to the Top-160 reranker candidates,
- score each candidate with the frozen classifier,
- select the highest-probability candidate per query,
- report prediction coverage and accuracy on the final evaluation set.

This stage is for prediction only. No retraining, no tuning, and no leakage are allowed.